In [90]:
import torch
import lmdb, pickle
lmdb_path = '/media/liud/Liud_FX2T/dataset/OC20/is2re_10k/select_data/all_stru/all_edge/train'



mean = torch.mean(tensor, dim=0).to(device)
std = torch.std(tensor, dim=0).to(device)

DataBatch(edge_index=[2, 1578], y=[1], pos=[52, 3], y_init=[1], y_relaxed=[1], pos_relaxed=[52, 3], force=[52, 3], atomic_numbers=[52], natoms=[1], tags=[52], fixed=[52], distances=[1844], cell_offsets=[1578, 3], cell=[1, 3, 3], sid=[1], ads_id=[1], batch=[52], ptr=[2], v1=[1578, 3], v1_num=1578, neighbors=[1], id_swap=[1578])


In [12]:
import numpy as np
import matplotlib.pyplot as plt
import lmdb
import pickle
import torch
import statistics

npz_path = '/home/liud/Documents/code/ocp/results/is2rv/2024-12-18-22-24-00/is2rv_predictions_0.npz'
test_path = '/media/liud/Liud_FX2T/dataset/OC20/is2re_100k/select_data/all_stru/50k/vector/all_edge/test'

# 读取 npz 文件中的数据
data = np.load(npz_path)
pos_predict = data['vector']
# ids = torch.tensor(data['ids'], dtype=int8)
sid_pre = data['ids']
sid_pre = torch.tensor(sid_pre.astype('int32'))

num_edges = torch.zeros_like(sid_pre)

env = lmdb.open(test_path, readonly=True)
sid_ori = []
num_edges_ori = []
pos_ori = []
# # 注意可能数据顺序不对
with env.begin(write=False) as txn: 
    cursor = txn.cursor()
    for key, value in cursor:
        info = pickle.loads(value)
        tags = info.tags
        tags12 = torch.where(tags > 0)[0]
        src, dst = info.edge_index
        cond1 = torch.isin(src, tags12)
        cond2 = torch.isin(dst, tags12)
        # 结合所有条件，生成一个布尔掩码
        mask = cond1 | cond2
        v = info.v1[mask]
        pos_ori.extend(v.tolist())
        sid_ori.append(info.sid[0])
        num_edges_ori.append(v.shape[0])
        sid = torch.where(sid_pre==info.sid[0])[0]
        num_edges[sid] = v.shape[0]
env.close()

sid_pre = torch.repeat_interleave(sid_pre, num_edges)
sorted_ids_pre, indices_pre = torch.sort(sid_pre)
pos_pre = torch.tensor(pos_predict)[indices_pre]

ids_ori = torch.repeat_interleave(torch.tensor(sid_ori), 
                                  torch.tensor(num_edges_ori))
sorted_ids_ori, indices_ori = torch.sort(ids_ori)
pos_ori = torch.tensor(pos_ori)[indices_ori]

mae = torch.mean(torch.abs(pos_pre - pos_ori))
# # mae = np.mean(np.abs(pos_predict - pos_relax), axis=0) # 按照坐标轴来取值
print(mae)

tensor(0.2055)


In [27]:
import numpy as np
import matplotlib.pyplot as plt
import lmdb
import pickle
import torch
import statistics

npz_path = '/home/liud/Documents/code/ocp/results/is2rve/2024-12-21-18-42-08/is2rve_predictions_0.npz'
test_path = '/media/liud/Liud_FX2T/dataset/OC20/is2re_10k/select_data/tag1&2/vector/test'

# 读取 npz 文件中的数据
data = np.load(npz_path)
pos_predict = data['vector']
energy_pre = data['energy']

# ids = torch.tensor(data['ids'], dtype=int8)
sid_pre = data['ids']
sid_pre = torch.tensor(sid_pre.astype('int32'))

num_edges = torch.zeros_like(sid_pre)

env = lmdb.open(test_path, readonly=True)
sid_ori = []
num_edges_ori = []
pos_ori = []
energy_ori = []
# # 注意可能数据顺序不对
with env.begin(write=False) as txn: 
    cursor = txn.cursor()
    for key, value in cursor:
        info = pickle.loads(value)
        tags = info.tags
        tags12 = torch.where(tags > 0)[0]
        src, dst = info.edge_index
        cond1 = torch.isin(src, tags12)
        cond2 = torch.isin(dst, tags12)
        # 结合所有条件，生成一个布尔掩码
        mask = cond1 | cond2
        v = info.v1[mask]
        pos_ori.extend(v.tolist())
        sid_ori.append(info.sid[0])
        num_edges_ori.append(v.shape[0])
        energy_ori.append(info.y) # 能量
        sid = torch.where(sid_pre==info.sid[0])[0]
        num_edges[sid] = v.shape[0]
env.close()
# 先处理energy排序
sorted_ids_pre, indices_pre = torch.sort(sid_pre)
sorted_ids_ori, indices_ori = torch.sort(torch.tensor(sid_ori))
energy_pre = torch.tensor(energy_pre)[indices_pre]
energy_ori = torch.tensor(energy_ori)[indices_ori]

sid_pre = torch.repeat_interleave(sid_pre, num_edges)
sorted_ids_pre, indices_pre = torch.sort(sid_pre)
pos_pre = torch.tensor(pos_predict)[indices_pre]

ids_ori = torch.repeat_interleave(torch.tensor(sid_ori), 
                                  torch.tensor(num_edges_ori))
sorted_ids_ori, indices_ori = torch.sort(ids_ori)
pos_ori = torch.tensor(pos_ori)[indices_ori]

pos_mae = torch.mean(torch.abs(pos_pre - pos_ori))
# # mae = np.mean(np.abs(pos_predict - pos_relax), axis=0) # 按照坐标轴来取值
energy_mae = torch.mean(torch.abs(energy_pre - energy_ori))
print('pos_mae:',pos_mae)
print('energy_mae:',energy_mae)

pos_mae: tensor(0.7016)
energy_mae: tensor(0.7234, dtype=torch.float64)
